# 29B Actions, approvals and the audit ledger
Reading and drafting are free; anything that leaves the firm, or changes what a client relies on, needs Priya's approval of the exact payload. Standing rules live in the server.

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
others = policy.confidential_terms(MANDATES, "mandate_a")
base = {"draft_id": "d-001", "scope": "mandate_a", "to": ["cio@harborstate.example"], "subject": "Juliett after its 10-Q",
        "body": "Juliett marked Crestline Roofing down 7.5 points, to 90.0, in its June 10-Q (0009000014-26-000141).", "attachments": []}
cases = {"the reply Priya would approve": {},
         "answers the question about Northgate": {"body": base["body"] + " Northgate has been adding to Hotel."},
         "obfuscated account code of the other client": {"body": base["body"] + " Ref NG\u200b-PC 9902."},
         "names a restricted issuer": {"body": base["body"] + " Ironwood Data Systems is refinancing."},
         "writes to the borrower": {"to": ["cfo@crestlineroofing.example"]},
         "copies an unknown address": {"to": ["cio@harborstate.example", "me@gmail.example"]}}
for name, change in cases.items():
    print(f"{name:45s} -> {policy.policy_check({**base, **change}, MANDATES['mandate_a'], REG, others) or 'allowed'}")

In [ ]:
t0 = NOW.timestamp()
tok = approvals.issue_token(base, "priya", now=t0, ttl_min=P["approvals"]["ttl_minutes"])
print("hash:", approvals.payload_hash(base)[:16], "| token:", tok[:24] + "...")
print("same payload        ->", approvals.verify_token(base, tok, "priya", t0 + 60))
print("recipient added     ->", approvals.verify_token({**base, "to": base["to"] + ["credit.pm@harborstate.example"]}, tok, "priya", t0 + 60))
print("31 minutes later    ->", approvals.verify_token(base, tok, "priya", t0 + 31 * 60))
print("presented for Tom   ->", approvals.verify_token(base, tok, "tom", t0 + 60))

In [ ]:
# watchlist_write is an INTERNAL_WRITE with an approval override: clients rely on the watchlist.
entry = {"draft_id": "w-001", "scope": "mandate_b", "bdc": "FIXTURE HOTEL SENIOR LENDING",
         "reason": "Brightwater down 5.5 points; Oakmont PIK up from 1.00% to 3.00% (0009000012-26-000121)"}
print("policy for watchlist_write:", catalog.policy_for("watchlist_write"))
wtok = approvals.issue_token(entry, "priya", now=t0)
print("approved entry ->", approvals.verify_token(entry, wtok, "priya", t0 + 5),
      "| reason edited afterwards ->", approvals.verify_token({**entry, "reason": "fine"}, wtok, "priya", t0 + 5))

In [ ]:
led = audit.Ledger(path=None, clock=clock.clock)
led.record("tool_call", tool="draft_email", scope="mandate_a", decision="executed", payload_hash=approvals.payload_hash(base))
led.record("approval_requested", tool="send_email", payload_hash=approvals.payload_hash(base))
led.record("approval_granted", approver="priya", payload_hash=approvals.payload_hash(base))
swapped = {**base, "to": base["to"] + ["credit.pm@harborstate.example"]}
led.record("policy_block", tool="send_email", rule_id="R-TOKEN", decision="blocked",
           reason=approvals.verify_token(swapped, tok, "priya", t0 + 60), payload_hash=approvals.payload_hash(swapped))
print(pd.DataFrame(led.events).fillna("")[["event_id", "ts", "kind", "tool", "decision", "rule_id", "reason"]])
print("\nThe ledger stores hashes and ids, never the email body.")

**Your turn.** Build `draft_email`, `request_approval`, `send_email` and `watchlist_write` in your server, with the policy engine reading action classes from `params.yaml`. Write tests for every blocked case in Part 6, and prove a correctly approved send reaches the mock outbox: a system that blocks everything fails.